In [ ]:
# dripyto# install basical image libs
!pip install Pillow>=5.0.0
!pip install -U image
!pip install wheel

# install torch and torchvision (a utility library for computer vision that provides many public datasets and pre-trained models)
!pip install torch torchvision

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 99.2 MB/s eta 0:00:00
  Created wheel for image: filename=image-1.5.33-py2.py3-none-any.whl size=19482 sha256=ef0ffebbed59b9b0d83e2ca56bd8729472db24e17bb3587164ee562338386c79
  Stored in directory: /root/.cache/pip/wheels/58/30/d8/3212cd83eeeeee0a1f0c7b9b7bd0674a2b9f09342870473a2a
Successfully built image


In [ ]:
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
cuda0 = torch.device('cuda:0')  # pick the GPU at index 0

In [ ]:
# make data
def unkn_fn(x):
  return 0.5*pow(x[:, 0],2) + 0.3*x[:,0]*x[:, 1] + 0.2*pow(x[:, 1],2)

# create input-output tuples for training and test sets
train_size = 1000
train_x = 2*torch.randn(train_size, 2, device=cuda0)-1  # this has to be on GPU too
train_target = unkn_fn(train_x)
print(train_x.shape, train_target.shape)

test_size = 1000
test_x = 2*torch.randn(test_size, 2, device=cuda0)-1  # this has to be on GPU too
test_target = unkn_fn(test_x)

train_target = train_target.reshape((train_size,1))
test_target = test_target.reshape((test_size,1))


torch.Size([1000, 2]) torch.Size([1000])


In [ ]:
class ThreeLayerNet(nn.Module):
    """3-layer network: Input(2) -> Hidden1 -> Hidden2 -> Output(1)"""
    def __init__(self, hidden1_size=64, hidden2_size=32):
        super(ThreeLayerNet, self).__init__()
        self.fc1 = nn.Linear(2, hidden1_size)
        self.fc2 = nn.Linear(hidden1_size, hidden2_size)
        self.fc3 = nn.Linear(hidden2_size, 1)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)  # No activation on output for regression
        return x

In [ ]:
class FourLayerNet(nn.Module):
    """4-layer network: Input(2) -> Hidden1 -> Hidden2 -> Hidden3 -> Output(1)"""
    def __init__(self, hidden1_size=64, hidden2_size=48, hidden3_size=24):
        super(FourLayerNet, self).__init__()
        self.fc1 = nn.Linear(2, hidden1_size)
        self.fc2 = nn.Linear(hidden1_size, hidden2_size)
        self.fc3 = nn.Linear(hidden2_size, hidden3_size)
        self.fc4 = nn.Linear(hidden3_size, 1)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        x = self.fc4(x)  # No activation on output for regression
        return x


In [ ]:
# Construct a network and move to GPU
net3 = ThreeLayerNet().to(cuda0)
net4 = FourLayerNet().to(cuda0)

# Construct an optimizer
optim3 = torch.optim.SGD(net3.parameters(), lr=0.01)
optim4 = torch.optim.SGD(net4.parameters(), lr=0.01)

In [ ]:
# Training loop

# TODO: train on training data ONLY!
for iter in range(5000):
    # clear gradients accumulated on the parameters
    net3.train()
    optim3.zero_grad()

    # TODO: forward pass
    result = net3(train_x)

    # TODO: compute loss
    loss = F.mse_loss(result, train_target)

    # compute gradients
    loss.backward()

    # let the optimizer do its work; the parameters will be updated in this call
    optim3.step()

    # add some printing
    if iter % 500 == 0:
        print('iteration {}\tloss {:.5f}'.format(iter, loss))


iteration 0	loss 34.42474
iteration 500	loss 0.27015
iteration 1000	loss 0.08047
iteration 1500	loss 0.05622
iteration 2000	loss 0.01859
iteration 2500	loss 0.01778
iteration 3000	loss 0.01531
iteration 3500	loss 0.01143
iteration 4000	loss 0.01085
iteration 4500	loss 0.00790


In [ ]:
# Test
# TODO: now test on test data and compare to true values
test_result = net3(test_x)
loss = F.mse_loss(test_result, test_target)

print('Loss over entire test set: ' + repr(loss))
vis = np.hstack((test_result.cpu().detach().numpy(), test_target.cpu().detach().numpy()))
print('   Predicted       True')
print(vis[:20])

Loss over entire test set: tensor(0.0248, device='cuda:0', grad_fn=<MseLossBackward0>)
   Predicted       True
[[ 1.3562694   1.3335378 ]
 [ 3.6281438   3.6437612 ]
 [ 6.772114    6.8336897 ]
 [ 2.789249    2.7565517 ]
 [ 1.206828    1.2246209 ]
 [ 2.5260608   2.5214355 ]
 [19.784231   19.780367  ]
 [ 3.784569    3.8401225 ]
 [ 4.082855    4.088606  ]
 [ 5.8022237   5.914355  ]
 [ 0.9261185   0.86908835]
 [ 0.1099669   0.07446127]
 [ 5.7294846   5.7755218 ]
 [ 3.1432579   3.134244  ]
 [ 5.1764207   5.3139687 ]
 [ 5.5082507   5.611217  ]
 [ 1.1106067   1.1648856 ]
 [ 4.6438723   4.487565  ]
 [ 1.1241661   1.1401188 ]
 [17.187855   17.219814  ]]


In [ ]:
# Training loop

# TODO: train on training data ONLY!
for iter in range(5000):
    # clear gradients accumulated on the parameters
    net4.train()
    optim4.zero_grad()

    # TODO: forward pass
    result = net4(train_x)

    # TODO: compute loss
    loss = F.mse_loss(result, train_target)

    # compute gradients
    loss.backward()

    # let the optimizer do its work; the parameters will be updated in this call
    optim4.step()

    # add some printing
    if iter % 500 == 0:
        print('iteration {}\tloss {:.5f}'.format(iter, loss))


iteration 0	loss 35.14498
iteration 500	loss 0.55382
iteration 1000	loss 0.26581
iteration 1500	loss 0.14494
iteration 2000	loss 0.08228
iteration 2500	loss 0.05391
iteration 3000	loss 0.04500
iteration 3500	loss 0.02903
iteration 4000	loss 0.02406
iteration 4500	loss 0.02297


In [ ]:
# Test
# TODO: now test on test data and compare to true values
test_result = net4(test_x)
loss = F.mse_loss(test_result, test_target)

print('Loss over entire test set: ' + repr(loss))
vis = np.hstack((test_result.cpu().detach().numpy(), test_target.cpu().detach().numpy()))
print('   Predicted       True')
print(vis[:20])

Loss over entire test set: tensor(0.0265, device='cuda:0', grad_fn=<MseLossBackward0>)
   Predicted       True
[[ 1.4057741   1.3335378 ]
 [ 3.7370973   3.6437612 ]
 [ 7.005792    6.8336897 ]
 [ 2.8102412   2.7565517 ]
 [ 1.2885921   1.2246209 ]
 [ 2.5969446   2.5214355 ]
 [20.367838   19.780367  ]
 [ 3.9642558   3.8401225 ]
 [ 4.1141305   4.088606  ]
 [ 6.0766225   5.914355  ]
 [ 0.77479064  0.86908835]
 [ 0.14574303  0.07446127]
 [ 5.9477606   5.7755218 ]
 [ 3.2122068   3.134244  ]
 [ 5.410759    5.3139687 ]
 [ 5.773703    5.611217  ]
 [ 1.2238419   1.1648856 ]
 [ 4.9186187   4.487565  ]
 [ 1.1562707   1.1401188 ]
 [17.589962   17.219814  ]]
